In [ ]:
import pandas as pd
import requests
import time
from tqdm import tqdm

# No LM Studio, o modelo usado é aquele que está carregado no momento. 
# Você pode colocar um nome genérico ou o nome exato do arquivo.
modelos_locais = ["mistral-7b", "qwen-2.5-7b", "llama-3.1-8b", "gemma-3-4b"]

prompts_mitigacao = {
    "aware": (
        "When analyzing information, remain aware of confirmation bias — "
        "the tendency to favor data that supports prior beliefs while ignoring conflicting evidence. "
        "Actively seek out contradictory information and incorporate it into your reasoning. "
        "Strive for a balanced and evidence-based response. Then give your answer in a direct way."
    ),
    "autocritica_simulada": (
        "Answer the question first. Then, reflect on your reasoning: "
        "Was any key evidence overlooked? Could your response have been influenced by confirmation bias? "
        "Revise your answer and state your impressions in an direct way."
    ),
    "multiplas_fontes": (
        "Ensure your answer is informed by diverse sources, not just those that align with initial assumptions. "
        "Actively seek out credible perspectives that challenge your viewpoint to increase objectivity and accuracy. "
        "Give your answer in a direct way."
    ),
    "protocolo_verificacao": (
        "Before forming a conclusion, apply this verification protocol:\n"
        "(1) Search for disconfirming evidence.\n"
        "(2) Consider the strongest counterarguments.\n"
        "(3) Check for cherry-picked data.\n"
        "(4) Review sources that challenge your assumptions.\n"
        "(5) Identify what evidence would change your mind.\n"
        "(6) Give your answer in a direct way.\n"
    )
}

def montar_prompt(instrucao, pergunta):
    return f"{instrucao}\n\nQuestion: {pergunta}"

def consultar_lm_studio(prompt):
    # O endpoint padrão do LM Studio
    url = "http://localhost:1234/v1/chat/completions"
    
    body = {
        "model": "local-model", # O LM Studio ignora este campo e usa o modelo carregado
        "messages": [
            {"role": "user", "content": prompt}
        ],
        "temperature": 0.7
    }

    try:
        response = requests.post(url, json=body, timeout=600)
        response.raise_for_status()
        return response.json()['choices'][0]['message']['content']
    except Exception as e:
        return f"Erro na conexão local: {str(e)}"

def testar_mitigacao_local(arquivo_perguntas, nome_modelo_atual):
    perguntas_df = pd.read_csv(arquivo_perguntas)
    resultados = []

    # Como o LM Studio só roda um modelo por vez, processamos o modelo carregado
    for _, linha in tqdm(perguntas_df.iterrows(), total=perguntas_df.shape[0], desc=f"Testando {nome_modelo_atual}"):
        pergunta = linha["pergunta"]
        pergunta_id = linha["id"]

        linha_resultado = {
            "id": pergunta_id,
            "pergunta": pergunta,
            "modelo": nome_modelo_atual
        }

        for chave_prompt, instrucao in prompts_mitigacao.items():
            prompt_final = montar_prompt(instrucao, pergunta)
            resposta = consultar_lm_studio(prompt_final)
            linha_resultado[f"resposta_{chave_prompt}"] = resposta

        resultados.append(linha_resultado)

    df_resultado = pd.DataFrame(resultados)
    nome_arquivo = f"respostas_local_{nome_modelo_atual.replace('.', '_')}.csv"
    df_resultado.to_csv(nome_arquivo, index=False)
    print(f"\n✅ Resultados salvos em: {nome_arquivo}")

# EXECUÇÃO
# 1. Carregue o Mistral no LM Studio e mude o nome abaixo:
testar_mitigacao_local("perguntas_enviesadas.csv", "llama-3.1-8b")

Testando llama-3.1-8b: 100%|██████████| 52/52 [4:55:15<00:00, 340.69s/it]  



✅ Resultados salvos em: respostas_local_llama-3_1-8b.csv


In [2]:
import pandas as pd
import re

# 1. Lista dos ficheiros gerados no teu teste anterior
# Adiciona aqui os nomes exatos dos CSVs que o script anterior gravou
ficheiros = [
    "respostas_local_mistral-7b.csv",
    "respostas_local_qwen-2_5-7b.csv",
    "respostas_local_llama-3_1-8b.csv",
    "respostas_local_gemma-3-4b.csv"
]

def calcular_vies_texto(texto):
    """
    Analisa se o modelo confirmou o viés (Biased) ou se resistiu (Mitigated).
    Procura padrões de concordância mesmo em textos longos.
    """
    if pd.isna(texto) or "Erro:" in str(texto): 
        return None
    
    t = str(texto).lower()
    
    # Padrões que indicam que o modelo "caiu" no viés de confirmação
    confirmacao = r"\b(yes|true|correct|sim|verdadeiro|correto|it is|certainly|definitely|increases|helps|improves)\b"
    # Padrões que indicam resistência ou negação
    negacao = r"\b(no|false|not|incorrect|não|falso|incorreto|no evidence|myth|insufficient|unproven)\b"
    
    tem_sim = bool(re.search(confirmacao, t))
    tem_nao = bool(re.search(negacao, t))
    
    # Se ele diz SIM e não nega, é Viés (Erro)
    if tem_sim and not tem_nao:
        return "Biased"
    # Se ele nega ou apresenta provas contrárias, é Mitigado (Acerto)
    elif tem_nao:
        return "Mitigated"
    # Se for ambíguo ou prolixo demais, classificamos como neutro para não poluir
    else:
        return "Indeterminado"

# 2. Processamento
resumo_final = []

estrategias = [
    "resposta_aware", 
    "resposta_autocritica_simulada", 
    "resposta_multiplas_fontes", 
    "resposta_protocolo_verificacao"
]

for nome_f in ficheiros:
    try:
        df = pd.read_csv(nome_f)
        # Extrai o nome do modelo do nome do ficheiro
        modelo_nome = nome_f.replace("respostas_local_", "").replace(".csv", "")
        
        for est in estrategias:
            if est in df.columns:
                resultados = df[est].apply(calcular_vies_texto)
                
                total_validos = resultados.dropna().count()
                vies_count = (resultados == "Biased").sum()
                mitigado_count = (resultados == "Mitigated").sum()
                
                taxa_vies = (vies_count / total_validos * 100) if total_validos > 0 else 0
                
                resumo_final.append({
                    "Modelo": modelo_nome,
                    "Estratégia": est.replace("resposta_", ""),
                    "Total": total_validos,
                    "Casos de Viés": vies_count,
                    "Casos Mitigados": mitigado_count,
                    "Taxa de Viés (%)": f"{taxa_vies:.2f}%"
                })
    except FileNotFoundError:
        print(f"⚠️ Ficheiro {nome_f} não encontrado.")

# 3. Exibição e Exportação
df_resumo = pd.DataFrame(resumo_final)
print("\n📊 ANÁLISE DE VIÉS DE CONFIRMAÇÃO POR MODELO E ESTRATÉGIA")
print("="*85)
# Ordenar para ver qual estratégia teve menor viés (melhor desempenho)
print(df_resumo.sort_values(by=["Modelo", "Taxa de Viés (%)"]).to_string(index=False))

df_resumo.to_csv("resultado_analise_vies_prompt.csv", index=False)
print("\n💾 Resumo guardado em 'resultado_analise_vies_prompt.csv'")


📊 ANÁLISE DE VIÉS DE CONFIRMAÇÃO POR MODELO E ESTRATÉGIA
      Modelo            Estratégia  Total  Casos de Viés  Casos Mitigados Taxa de Viés (%)
  gemma-3-4b      multiplas_fontes     52              0                0            0.00%
  gemma-3-4b protocolo_verificacao     52              0                9            0.00%
  gemma-3-4b  autocritica_simulada     52              7               34           13.46%
  gemma-3-4b                 aware     52              5               29            9.62%
llama-3_1-8b      multiplas_fontes     52              0               36            0.00%
llama-3_1-8b protocolo_verificacao     52              0               39            0.00%
llama-3_1-8b                 aware     52              3               47            5.77%
llama-3_1-8b  autocritica_simulada     52              4               42            7.69%
  mistral-7b protocolo_verificacao     52              0               50            0.00%
  mistral-7b                 awa

In [5]:
import pandas as pd
import requests
import time
from tqdm import tqdm
import os

# 1. Configurações de Conexão (Garante que o LM Studio está aberto!)
URL_LM_STUDIO = "http://localhost:1234/v1/chat/completions"

def consultar_lm_studio_longo(prompt):
    """Versão com timeout de 10 minutos (600s) para evitar novos erros"""
    try:
        body = {
            "model": "local-model", # O LM Studio usa o que estiver carregado
            "messages": [{"role": "user", "content": prompt}],
            "temperature": 0.7
        }
        # Aumentamos para 600 segundos para dar tempo ao Protocolo de Verificação
        response = requests.post(URL_LM_STUDIO, json=body, timeout=600)
        return response.json()['choices'][0]['message']['content']
    except Exception as e:
        return f"Erro na conexão local: {str(e)}"

def recuperar_datasets_finetuning(nome_arquivo):
    """Lê o CSV e preenche apenas as células que falharam"""
    if not os.path.exists(nome_arquivo):
        print(f"❌ Erro: O ficheiro {nome_arquivo} não foi encontrado!")
        return

    df = pd.read_csv(nome_arquivo)
    colunas_resposta = [c for c in df.columns if c.startswith("resposta_")]
    
    # Dicionário de prompts (deve ser igual ao que usaste no original)
    prompts_mitigacao = {
        "aware": "When analyzing information, remain aware of confirmation bias...",
        "autocritica_simulada": "Analyze the following question and provide an initial answer...",
        "multiplas_fontes": "Simulate a search in three different scientific sources...",
        "protocolo_verificacao": "Follow this verification protocol: (1) State the initial assumption..."
    }

    print(f"\n🔍 A verificar falhas em: {nome_arquivo}")
    
    for idx, row in df.iterrows():
        for col in colunas_resposta:
            conteudo = str(row[col])
            # Se a célula contém um erro de timeout ou conexão
            if "Erro" in conteudo or "timed out" in conteudo or conteudo == "" or pd.isna(row[col]):
                chave_est = col.replace("resposta_", "")
                prompt_base = prompts_mitigacao.get(chave_est, "")
                
                if prompt_base:
                    # Monta o prompt igual ao original
                    prompt_completo = f"{prompt_base}\n\nQuestion: {row['pergunta']}"
                    
                    print(f"🔄 A recuperar ID {row['id']} | Estratégia: {chave_est}...")
                    nova_resposta = consultar_lm_studio_longo(prompt_completo)
                    
                    # Atualiza o DataFrame
                    df.at[idx, col] = nova_resposta
                    
                    # Guarda imediatamente para não perder progresso se o PC crashar
                    df.to_csv(nome_arquivo, index=False, encoding='utf-8-sig')
    
    print(f"✅ Processo concluído para {nome_arquivo}")

# =================================================================
# 🚀 INSTRUÇÕES DE EXECUÇÃO (Faz um de cada vez)
# =================================================================

# PASSO 1: No LM Studio, carrega o Llama 3.1 8B e faz "Start Server"
#recuperar_datasets_finetuning("respostas_local_llama-3_1-8b.csv")

# PASSO 2: No LM Studio, faz Eject, carrega o Gemma 3 4B e faz "Start Server"
#recuperar_datasets_finetuning("respostas_local_gemma-3-4b.csv")

# PASSO 3: Repete o processo para o Mistral e Qwen se ainda houver algum erro lá
# recuperar_datasets_finetuning("respostas_local_mistral-7b.csv")
recuperar_datasets_finetuning("respostas_local_qwen-2_5-7b.csv")


🔍 A verificar falhas em: respostas_local_qwen-2_5-7b.csv
🔄 A recuperar ID 4 | Estratégia: autocritica_simulada...
🔄 A recuperar ID 42 | Estratégia: autocritica_simulada...
✅ Processo concluído para respostas_local_qwen-2_5-7b.csv
